In [3]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Import MNIST dataset
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
X = X / 255.0   # normalize
y = y.astype(int)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# One-hot encoding for labels
def one_hot(y, num_classes=10):
    oh = np.zeros((len(y), num_classes))
    oh[np.arange(len(y)), y] = 1
    return oh

y_train_oh = one_hot(y_train)

# 2. Build a feedforward neural network (1 hidden layer)
input_size = 784
hidden_size = 64
output_size = 10
lr = 0.3
epochs = 100

# Initialize weights
W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

# Activation (ReLU + softmax)
def relu(x): return np.maximum(0, x)
def relu_deriv(x): return (x > 0).astype(float)
def softmax(x): 
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

# 3. Training (forward + backpropagation)
for epoch in range(epochs):
    # Forward pass
    z1 = X_train @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    a2 = softmax(z2)

    # Loss (cross-entropy)
    loss = -np.mean(np.sum(y_train_oh * np.log(a2 + 1e-8), axis=1))

    # Backpropagation
    dz2 = a2 - y_train_oh
    dW2 = a1.T @ dz2 / len(X_train)
    db2 = np.mean(dz2, axis=0, keepdims=True)

    dz1 = (dz2 @ W2.T) * relu_deriv(z1)
    dW1 = X_train.T @ dz1 / len(X_train)
    db1 = np.mean(dz1, axis=0, keepdims=True)

    # Update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

# 4. Evaluation
z1 = X_test @ W1 + b1
a1 = relu(z1)
z2 = a1 @ W2 + b2
a2 = softmax(z2)
y_pred = np.argmax(a2, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Epoch 1/100, Loss: 2.3031
Epoch 2/100, Loss: 2.3011
Epoch 3/100, Loss: 2.2991
Epoch 4/100, Loss: 2.2969
Epoch 5/100, Loss: 2.2943
Epoch 6/100, Loss: 2.2912
Epoch 7/100, Loss: 2.2875
Epoch 8/100, Loss: 2.2828
Epoch 9/100, Loss: 2.2771
Epoch 10/100, Loss: 2.2699
Epoch 11/100, Loss: 2.2609
Epoch 12/100, Loss: 2.2498
Epoch 13/100, Loss: 2.2360
Epoch 14/100, Loss: 2.2189
Epoch 15/100, Loss: 2.1980
Epoch 16/100, Loss: 2.1727
Epoch 17/100, Loss: 2.1425
Epoch 18/100, Loss: 2.1068
Epoch 19/100, Loss: 2.0654
Epoch 20/100, Loss: 2.0179
Epoch 21/100, Loss: 1.9642
Epoch 22/100, Loss: 1.9045
Epoch 23/100, Loss: 1.8389
Epoch 24/100, Loss: 1.7682
Epoch 25/100, Loss: 1.6935
Epoch 26/100, Loss: 1.6164
Epoch 27/100, Loss: 1.5387
Epoch 28/100, Loss: 1.4623
Epoch 29/100, Loss: 1.3888
Epoch 30/100, Loss: 1.3195
Epoch 31/100, Loss: 1.2550
Epoch 32/100, Loss: 1.1957
Epoch 33/100, Loss: 1.1414
Epoch 34/100, Loss: 1.0918
Epoch 35/100, Loss: 1.0468
Epoch 36/100, Loss: 1.0057
Epoch 37/100, Loss: 0.9683
Epoch 38/1

In [5]:
import numpy as np

# Dataset (XOR)
x = np.array([
    [0,0],
    [0,1],
    [1,0],
    [1,1]
])
y = np.array([[1],[0],[1],[0]])

# Hardcoded first layer (4 hidden neurons)
w1 = np.array([
    [-1,-1],
    [-1, 1],
    [ 1,-1],
    [ 1, 1]
])
b1 = np.array([[-1],[-1],[-1],[-1]])

# Trainable second layer
np.random.seed(42)
w2 = np.random.randn(4,1)
b2 = np.random.randn(1,1)

# Step activation
def activation(z):
    return np.where(z >= 0, 1, 0)

# Training loop
lr = 0.1
epochs = 1000
for epoch in range(epochs):
    # First layer (fixed)
    z1 = np.dot(x, w1.T) + b1.T
    a1 = activation(z1)

    # Second layer (trainable)
    z2 = np.dot(a1, w2) + b2
    a2 = activation(z2)

    # Error
    error = y - a2

    # Backprop only for layer 2
    d_a2 = error
    d_w2 = np.dot(a1.T, d_a2)
    d_b2 = np.sum(d_a2, axis=0, keepdims=True)

    # Update second layer weights
    w2 += lr * d_w2
    b2 += lr * d_b2

# Final forward pass
z1 = np.dot(x, w1.T) + b1.T
a1 = activation(z1)
z2 = np.dot(a1, w2) + b2
a2 = activation(z2)

print("Predicted:\n", a2)
print("Actual:\n", y)


Predicted:
 [[1]
 [0]
 [1]
 [0]]
Actual:
 [[1]
 [0]
 [1]
 [0]]
